# Microestructura de Mercado — BTC

Del precio como dato fijo al precio como resultado de una subasta continua.

## Objetivo de la sesion

- entender la estructura de un Limit Order Book (LOB): bids, asks, spread, mid price
- cargar datos reales de LOB y calcular metricas basicas con pandas
- visualizar profundidad y medir presion de mercado con el imbalance

Antes de ejecutar cada celda, intenta predecir que deberia salir.

## 0. El puente desde Lesson 3

En las clases anteriores usabais precios como datos fijos: `price = 100000`. Un numero que aparecia en un dict o en un constructor de `Order`.

Pero ese numero no sale de la nada. Sale de una **subasta continua** donde miles de compradores y vendedores colocan ordenes. El Limit Order Book (LOB) es esa subasta.

Hoy vamos a abrir la caja negra del precio.

## 1. Cargar los datos del LOB

Tenemos 500 snapshots del order book de BTCUSDT — uno por minuto, ~8 horas de mercado simulado. Cada snapshot tiene 10 niveles de bids y 10 de asks.

**Antes de ejecutar:** cuantas columnas esperas? (pista: timestamp + 10 niveles x 2 lados x 2 campos)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/btc_lob_snapshots.csv")

print("filas:", len(df))
print("columnas:", len(df.columns))
print()
print("primeras columnas:", list(df.columns[:7]))
print("...ultimas:", list(df.columns[-4:]))
df.head(3)

## 2. Anatomia de un snapshot

Cada fila es una foto del LOB en un instante. Vamos a desmontar la primera fila y extraer las metricas basicas.

**Antes de ejecutar:** el best bid es el precio mas alto que alguien ofrece comprar. El best ask es el mas bajo que alguien ofrece vender. Cual sera mayor?

In [ ]:
row = df.iloc[0]

best_bid = row["bid_price_1"]
best_ask = row["ask_price_1"]
spread = best_ask - best_bid
mid = (best_bid + best_ask) / 2

print(f"best bid:  ${best_bid:,.2f}  (alguien quiere COMPRAR a este precio)")
print(f"best ask:  ${best_ask:,.2f}  (alguien quiere VENDER a este precio)")
print(f"spread:    ${spread:,.2f}  (coste implicito de operar)")
print(f"mid price: ${mid:,.2f}  (referencia neutral)")
print()
print(f"Si compras y vendes inmediatamente, pierdes ${spread:,.2f} por el spread.")
print(f"En porcentaje: {spread / mid * 100:.4f}%")

## 3. Visualizar un snapshot del LOB

Vamos a pintar el order book como un grafico de barras horizontales: bids a la izquierda (verde), asks a la derecha (rojo). Asi se ve la estructura real del libro.

**Antes de ejecutar:** los precios de bid van de mayor a menor (nivel 1 = mejor). Los de ask van de menor a mayor. Que forma esperarias?

In [ ]:
row = df.iloc[0]

bid_prices = [row[f"bid_price_{i}"] for i in range(1, 11)]
bid_sizes  = [row[f"bid_size_{i}"]  for i in range(1, 11)]
ask_prices = [row[f"ask_price_{i}"] for i in range(1, 11)]
ask_sizes  = [row[f"ask_size_{i}"]  for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 6))

# Bids: barras hacia la izquierda (negativas)
ax.barh(bid_prices, [-s for s in bid_sizes], height=8, color="#4ade80", alpha=0.8, label="Bids")
# Asks: barras hacia la derecha
ax.barh(ask_prices, ask_sizes, height=8, color="#f87171", alpha=0.8, label="Asks")

ax.axhline(y=(bid_prices[0] + ask_prices[0]) / 2, color="#22d3ee", linestyle="--", linewidth=1, label="Mid price")
ax.set_xlabel("Volumen (BTC)")
ax.set_ylabel("Precio (USD)")
ax.set_title("Snapshot del LOB — BTCUSDT")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Depth chart — profundidad acumulada

El grafico anterior muestra volumen por nivel. Pero lo que importa para operar es el **volumen acumulado**: cuanto puedo comprar o vender antes de mover el precio X dolares.

**Antes de ejecutar:** la curva de bids deberia subir de derecha a izquierda (mas volumen acumulado lejos del mid). La de asks sube de izquierda a derecha.

In [ ]:
row = df.iloc[0]

bid_prices = [row[f"bid_price_{i}"] for i in range(1, 11)]
bid_sizes  = [row[f"bid_size_{i}"]  for i in range(1, 11)]
ask_prices = [row[f"ask_price_{i}"] for i in range(1, 11)]
ask_sizes  = [row[f"ask_size_{i}"]  for i in range(1, 11)]

# Volumen acumulado
bid_cum = []
ask_cum = []
total = 0
for s in bid_sizes:
    total += s
    bid_cum.append(total)

total = 0
for s in ask_sizes:
    total += s
    ask_cum.append(total)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(bid_prices, bid_cum, alpha=0.3, color="#4ade80", step="mid")
ax.step(bid_prices, bid_cum, color="#4ade80", linewidth=2, where="mid", label="Bids (acumulado)")
ax.fill_between(ask_prices, ask_cum, alpha=0.3, color="#f87171", step="mid")
ax.step(ask_prices, ask_cum, color="#f87171", linewidth=2, where="mid", label="Asks (acumulado)")

ax.axvline(x=(bid_prices[0] + ask_prices[0]) / 2, color="#22d3ee", linestyle="--", linewidth=1, label="Mid")
ax.set_xlabel("Precio (USD)")
ax.set_ylabel("Volumen acumulado (BTC)")
ax.set_title("Depth Chart — BTCUSDT")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Volumen total en bids (10 niveles): {bid_cum[-1]:.2f} BTC")
print(f"Volumen total en asks (10 niveles): {ask_cum[-1]:.2f} BTC")

## 5. Imbalance — presion compradora vs vendedora

El imbalance mide quien tiene mas peso en el libro. Formula:

```
imbalance = bid_vol / (bid_vol + ask_vol)
```

- `imbalance > 0.5` → mas volumen en bids → presion compradora
- `imbalance < 0.5` → mas volumen en asks → presion vendedora
- `imbalance = 0.5` → equilibrio

**Antes de ejecutar:** mira el snapshot anterior — habia mas volumen en bids o en asks?

In [ ]:
row = df.iloc[0]

# Imbalance con los 5 mejores niveles
bid_vol = sum(row[f"bid_size_{i}"] for i in range(1, 6))
ask_vol = sum(row[f"ask_size_{i}"] for i in range(1, 6))
imbalance = bid_vol / (bid_vol + ask_vol)

print(f"bid vol (top 5): {bid_vol:.4f} BTC")
print(f"ask vol (top 5): {ask_vol:.4f} BTC")
print(f"imbalance:       {imbalance:.4f}")
print()
if imbalance > 0.5:
    print("→ Presion compradora: mas volumen en bids que en asks.")
else:
    print("→ Presion vendedora: mas volumen en asks que en bids.")

## 6. Imbalance a lo largo del tiempo

Un snapshot es una foto. Pero la senal esta en la evolucion. Vamos a calcular el imbalance para los 500 snapshots y ver como cambia junto al mid price.

**Antes de ejecutar:** si el imbalance sube (mas presion compradora), que esperarias que haga el mid price?

In [ ]:
# Calcular metricas para todos los snapshots
df["best_bid"] = df["bid_price_1"]
df["best_ask"] = df["ask_price_1"]
df["spread"]   = df["best_ask"] - df["best_bid"]
df["mid"]      = (df["best_bid"] + df["best_ask"]) / 2

# Imbalance (top 5 niveles)
df["bid_vol_5"] = sum(df[f"bid_size_{i}"] for i in range(1, 6))
df["ask_vol_5"] = sum(df[f"ask_size_{i}"] for i in range(1, 6))
df["imbalance"] = df["bid_vol_5"] / (df["bid_vol_5"] + df["ask_vol_5"])

# Tiempo en minutos desde el inicio
df["minutes"] = (df["timestamp"] - df["timestamp"].iloc[0]) / 60

# Grafico dual: mid price + imbalance
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(df["minutes"], df["mid"], color="#22d3ee", linewidth=1)
ax1.set_ylabel("Mid Price (USD)")
ax1.set_title("Mid Price y Imbalance — BTCUSDT (500 snapshots)")
ax1.grid(alpha=0.3)

ax2.plot(df["minutes"], df["imbalance"], color="#fbbf24", linewidth=0.8, alpha=0.7)
ax2.axhline(y=0.5, color="#a1a1aa", linestyle="--", linewidth=0.8, label="Equilibrio")
ax2.fill_between(df["minutes"], 0.5, df["imbalance"],
                 where=df["imbalance"] > 0.5, color="#4ade80", alpha=0.2)
ax2.fill_between(df["minutes"], 0.5, df["imbalance"],
                 where=df["imbalance"] < 0.5, color="#f87171", alpha=0.2)
ax2.set_ylabel("Imbalance")
ax2.set_xlabel("Minutos")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Imbalance medio: {df['imbalance'].mean():.4f}")
print(f"Imbalance std:   {df['imbalance'].std():.4f}")

## 7. Spread a lo largo del tiempo

El spread no es constante. Se amplia cuando hay incertidumbre o poca liquidez, y se estrecha cuando el mercado esta tranquilo.

**Antes de ejecutar:** esperas picos puntuales (eventos de spread amplio) o un spread muy estable?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df["minutes"], df["spread"], color="#f472b6", linewidth=0.8)
ax.set_xlabel("Minutos")
ax.set_ylabel("Spread (USD)")
ax.set_title("Spread a lo largo del tiempo — BTCUSDT")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Spread medio:  ${df['spread'].mean():.2f}")
print(f"Spread min:    ${df['spread'].min():.2f}")
print(f"Spread max:    ${df['spread'].max():.2f}")
print(f"Spread > $30:  {(df['spread'] > 30).sum()} snapshots de {len(df)}")

## 8. Tu turno — Weighted mid price

El mid price clasico trata bid y ask como iguales: `(bid + ask) / 2`. Pero si hay mucho mas volumen en bids que en asks, el precio "real" esta mas cerca del ask (los compradores estan empujando).

El **weighted mid price** pondera por los volumenes del mejor nivel:

```
wmid = (bid_price * ask_size + ask_price * bid_size) / (bid_size + ask_size)
```

Calcula `df["wmid"]` para todos los snapshots y comparalo con `df["mid"]`.

**Pista:** usa `df["bid_price_1"]`, `df["ask_price_1"]`, `df["bid_size_1"]`, `df["ask_size_1"]`.

In [ ]:
df["wmid"] = None  # <- reemplaza esta linea con la formula

print("TODO -> calcula df['wmid'] usando la formula del weighted mid price")

## 9. Solucion — Weighted mid price

Comparala con tu enfoque. La clave es que el weighted mid se acerca al ask cuando hay mas volumen en bids.

In [ ]:
bp = df["bid_price_1"]
ap = df["ask_price_1"]
bs = df["bid_size_1"]
as_ = df["ask_size_1"]

df["wmid"] = (bp * as_ + ap * bs) / (bs + as_)

# Comparacion
diff = df["wmid"] - df["mid"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(df["minutes"], df["mid"], color="#a1a1aa", linewidth=0.8, label="Mid")
ax1.plot(df["minutes"], df["wmid"], color="#22d3ee", linewidth=0.8, label="Weighted Mid")
ax1.set_ylabel("Precio (USD)")
ax1.set_title("Mid vs Weighted Mid — BTCUSDT")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.bar(df["minutes"], diff, width=0.8, color="#818cf8", alpha=0.6)
ax2.axhline(y=0, color="#a1a1aa", linewidth=0.5)
ax2.set_xlabel("Minutos")
ax2.set_ylabel("wmid - mid (USD)")
ax2.set_title("Diferencia: positivo = presion compradora empuja wmid hacia el ask")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Diferencia media: ${diff.mean():.4f}")
print(f"Diferencia max:   ${diff.abs().max():.4f}")

## Cierre

Que deberia haberte quedado claro:

- **LOB:** una lista de ordenes de compra (bids) y venta (asks) organizadas por precio. Es la subasta continua que produce el precio.
- **Spread:** `ask - bid`. Es el coste implicito de operar. Se amplia con incertidumbre.
- **Profundidad:** volumen acumulado por nivel. Te dice cuanto puedes operar antes de mover el precio.
- **Imbalance:** `bid_vol / (bid_vol + ask_vol)`. Mide presion compradora vs vendedora.
- **Weighted mid:** pondera el mid price por volumen. Mas informativo que el mid simple.

**Siguiente paso:** ya entiendes la estructura del LOB. En Lesson 5 veremos que pasa cuando envias una orden — tipos de ordenes (market, limit, stop) y como el matching engine las procesa.